# Active Learning NLI on Colab

**Edit the one cell below, then `Runtime -> Run all`.** Everything after it is derived
from the config file you point at -- data staging, mixed precision, output paths,
plots. Switching between a smoke test and a full run is a one-line change.

Runtime -> Change runtime type -> **A100 GPU** + **High-RAM**.

The repo lives in Drive and results are written to `models/` inside it, so they
survive a disconnect.

## Configuration -- the only cell you need to edit

In [ ]:
# Path to the repo inside Drive
REPO_DIR = '/content/drive/MyDrive/AL-NLI/nli-training-example'

# The experiment to run. Everything downstream is read from this file.
#   colab/params/model-xlm__strategy-NegE__smoke.json          ~10 min, validates the pipeline
#   colab/params/model-xlm__strategy-Rem__smoke.json           ~10 min
#   colab/params/model-xlm__strategy-NegE__colab_fulldata.json ~15 h  (needs High-RAM)
#   colab/params/model-xlm__strategy-Rem__colab_fulldata.json  ~12 h  (needs High-RAM)
#   colab/params/model-xlm__strategy-NegE__colab.json          ~15 h  (uses uploaded data_colab/)
CONFIG_PATH = 'colab/params/model-xlm__strategy-NegE__smoke.json'

# Where the dataset lives inside Drive, staged to local disk when the config asks
# for an absolute dataset_folder. Only used for configs pointing at /content/...
SOURCE_DATA_SUBDIR = 'data'

# 'auto'  -> stage the whole train.json for real runs; only the first 3x max_samples
#            rows for small smoke configs, so a smoke run starts in seconds.
# int     -> stage exactly this many rows.  None -> always stage the whole file.
STAGE_TRAIN_ROWS = 'auto'

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

assert os.path.isdir(REPO_DIR), (
    f'{REPO_DIR} not found. Upload the repo to Drive, or clone it there:\n'
    f'  !git clone <your-repo-url> "{REPO_DIR}"'
)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)          # so `import esnlir` resolves to the Drive copy
assert os.path.isfile(CONFIG_PATH), f'{CONFIG_PATH} not found under {REPO_DIR}'
print('repo  :', os.getcwd())
print('config:', CONFIG_PATH)

In [ ]:
# Colab already ships a CUDA build of torch -- do NOT reinstall it.
!pip install -q "transformers>=4.53,<5" "accelerate>=1.8" "scikit-learn>=1.7" tensorboard

import torch, transformers, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)

In [ ]:
import torch, psutil, os

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> A100 GPU.')

GPU_NAME = torch.cuda.get_device_name(0)
VRAM     = torch.cuda.get_device_properties(0).total_memory / 1e9
RAM      = psutil.virtual_memory().total / 1e9
CAP      = torch.cuda.get_device_capability(0)
BF16_OK  = CAP[0] >= 8                      # Ampere or newer
N_CPU    = os.cpu_count()

print(f'GPU  : {GPU_NAME}  ({VRAM:.0f} GB VRAM, compute {CAP[0]}.{CAP[1]})')
print(f'RAM  : {RAM:.0f} GB')
print(f'CPUs : {N_CPU}')
print(f'bf16 : {"yes" if BF16_OK else "no -- will fall back to fp16"}')
if 'A100' not in GPU_NAME:
    print('\nNOTE: the full configs are sized for an A100; expect several times longer here.')

## Resolve the config for this runtime

Mixed precision and worker count depend on the machine you actually got, so they are
patched into a resolved copy rather than edited in your config file. The run uses the
resolved copy; the original is never modified.

In [ ]:
import json, os

with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)
CFG_SOURCE = CONFIG_PATH        # later cells assert against this to catch stale state

# bf16 needs Ampere+; fall back to fp16 so `Run all` works on any GPU
if cfg.get('bf16') and not BF16_OK:
    cfg['bf16'], cfg['fp16'] = False, True
    print('bf16 unsupported here -> using fp16')
cfg['dataloader_num_workers'] = min(cfg.get('dataloader_num_workers', 4), max(1, N_CPU - 1))

STRATEGY = cfg['al_strategy']
OUT_ROOT = os.path.join(
    cfg['output_folder'],
    f"{cfg['model_type'].split('/')[-1]}_active_{STRATEGY}"
)
RUN_NAME = os.path.splitext(os.path.basename(CONFIG_PATH))[0]
LOG      = f'logs/{RUN_NAME}.log'
RESOLVED = f'logs/{RUN_NAME}.resolved.json'
os.makedirs('logs', exist_ok=True)
with open(RESOLVED, 'w') as fh:
    json.dump(cfg, fh, indent=2)

print(f'config     : {CFG_SOURCE}')
print(f'strategy   : {STRATEGY}')
print(f'data       : {cfg["dataset_folder"]}   max_samples={cfg["max_samples"]}')
print(f'batch/eval : {cfg["batch_size"]} / {cfg["eval_batch_size"]}   '
      f'bf16={cfg.get("bf16")} fp16={cfg.get("fp16")} workers={cfg["dataloader_num_workers"]}')
print(f'al_K       : {cfg["al_K"]}  seed_set={cfg.get("al_seed_size")}  remove_k={cfg.get("al_remove_k")}')
print(f'outputs -> {OUT_ROOT}')
print(f'log     -> {LOG}')

## Data

If the config asks for an absolute `dataset_folder` (e.g. `/content/data`) the dataset
is staged from Drive to local disk. Training re-reads every example each epoch --
`BERTDataset` tokenises inside `__getitem__` -- so leaving it on the Drive FUSE mount
means repeated slow I/O for the whole run, not a one-off read.

In [ ]:
import os, shutil, time, itertools

# Guard against stale kernel state: `cfg` is set several cells up, so editing
# CONFIG_PATH without re-running that cell would silently stage the wrong dataset.
assert CFG_SOURCE == CONFIG_PATH, (
    f'Stale config. cfg was loaded from {CFG_SOURCE!r} but CONFIG_PATH is now '
    f'{CONFIG_PATH!r}.\nRe-run the "Resolve the config" cell, or Runtime -> Run all.'
)

DATA_DIR = cfg['dataset_folder']
FILES = ['train.json', 'val.json', 'test.json', 'test_full.jsonl']
print(f'{CONFIG_PATH} -> dataset_folder = {DATA_DIR}')

if not os.path.isabs(DATA_DIR):
    # Config points at a folder already inside the repo (e.g. data_colab/)
    assert os.path.isdir(DATA_DIR), (
        f'{CONFIG_PATH} asks for {DATA_DIR}/, which is not in the repo.\n'
        f'Either build it locally with  python colab/prepare_colab_data.py  and upload it,\n'
        f'or set CONFIG_PATH to a *_colab_fulldata.json config (reads data/ from Drive).'
    )
    print(f'using {DATA_DIR}/ from the repo (no staging)')
else:
    src_dir = os.path.join(REPO_DIR, SOURCE_DATA_SUBDIR)
    assert os.path.isdir(src_dir), f'{src_dir} not found in Drive'
    os.makedirs(DATA_DIR, exist_ok=True)

    # How much of train.json to stage
    rows = STAGE_TRAIN_ROWS
    if rows == 'auto':
        ms = cfg.get('max_samples')
        # Small smoke configs only need headroom for the stratified subsample.
        # Real runs stage the whole file so the sampled subset matches an HPC run.
        rows = ms * 3 if (ms and ms < 500_000) else None
    print(f'staging train.json: {"whole file" if rows is None else f"first {rows:,} rows"}')

    for fname in FILES:
        src, dst = os.path.join(src_dir, fname), os.path.join(DATA_DIR, fname)
        if not os.path.exists(src):
            print(f'  {fname}: not in Drive, skipped'); continue
        limit = rows if fname == 'train.json' else None
        if os.path.exists(dst) and limit is None and os.path.getsize(dst) == os.path.getsize(src):
            print(f'  {fname}: already staged'); continue
        t0 = time.time()
        if limit is None:
            shutil.copyfile(src, dst)
        else:
            with open(src, 'rb') as fin, open(dst, 'wb') as fout:
                fout.writelines(itertools.islice(fin, limit))
        mb, dt = os.path.getsize(dst) / 1e6, max(time.time() - t0, 1e-6)
        print(f'  {fname}: {mb:,.0f} MB in {dt:.0f}s ({mb/dt:.0f} MB/s)')

for fname in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, fname)
    with open(path, 'rb') as fh:
        n = sum(1 for _ in fh)
    print(f'{fname:<20} {n:>9,} rows  {os.path.getsize(path)/1e6:>8.1f} MB')

In [ ]:
# Rough runtime estimate. Throughput is for an A100 + bf16 @ seq256;
# the run cell reports measured minutes-per-cycle once cycles complete.
TRAIN_PER_S, INFER_PER_S = (300, 1100) if 'A100' in GPU_NAME else (60, 220)

pool = cfg.get('max_samples') or sum(1 for _ in open(os.path.join(DATA_DIR, 'train.json')))
pool = min(pool, sum(1 for _ in open(os.path.join(DATA_DIR, 'train.json'))))
val  = sum(1 for _ in open(os.path.join(DATA_DIR, 'val.json')))
K, seed = cfg['al_K'], cfg.get('al_seed_size', 0)
rk, ep  = cfg.get('al_remove_k') or 0, cfg['n_epochs']

unl, lab, passes, score, cycles = pool - seed, seed, seed * ep, 0, 0
while unl > 0:
    score += unl
    unl -= min(rk, unl)
    a = min(K, unl); unl -= a; lab += a; cycles += 1; passes += lab * ep

hours = passes/TRAIN_PER_S/3600 + (score + val*cycles)/INFER_PER_S/3600
print(f'pool {pool:,} | {cycles} cycles | {passes/1e6:.2f}M train-passes, {score/1e6:.2f}M scoring')
print(f'estimate ~{hours:.1f} h  ({"fits" if hours < 24 else "TOO LONG"} in a 24 h session)')

## Run

Runs in the foreground and streams output live, so `Run all` works end to end and you
can watch progress here. The same output is written to the log in Drive. tqdm bars are
throttled to one line every 30 s to keep this readable.

With Pro+ background execution the run survives closing the tab.

In [ ]:
import subprocess, sys, re, time, os

TQDM = re.compile(r'\d+%\|')
started = time.time()
last_bar = 0.0

# `esnlir` is not pip-installed here, and running train.py as a script puts
# esnlir/training/ on sys.path -- not the repo root. PYTHONPATH fixes the import
# for the child process (sys.path.insert above only affects this kernel).
env = dict(os.environ)
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')
env['TOKENIZERS_PARALLELISM'] = 'false'      # silences the fork warning from num_workers>0

proc = subprocess.Popen(
    [sys.executable, '-u', 'esnlir/training/train.py', '--config-file', RESOLVED],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env
)
with open(LOG, 'w', encoding='utf-8') as log:
    for raw in proc.stdout:
        log.write(raw); log.flush()
        line = raw.rstrip('\n')
        if TQDM.search(line):
            if time.time() - last_bar > 30:
                last_bar = time.time()
                print(f'[{(time.time()-started)/60:6.1f}m] {line.split(chr(13))[-1][:110]}')
            continue
        if line.strip():
            print(line)

rc = proc.wait()
print(f'\n=== finished rc={rc} after {(time.time()-started)/3600:.2f} h -> {LOG} ===')
if rc != 0:
    print('non-zero exit; last lines of the log:')
    print(''.join(open(LOG).readlines()[-30:]))

## Results

In [ ]:
# Did the run actually produce anything? OUT_ROOT is created as the first statement
# of ActiveLearningTrainer.run(), so if it is missing the failure was during setup.
import os

HAVE_OUTPUT = os.path.isdir(OUT_ROOT)
if not HAVE_OUTPUT:
    print(f'{OUT_ROOT} does not exist -- the run never reached the AL loop.\n')
    if os.path.exists(LOG):
        print(f'last 40 lines of {LOG}:\n')
        print(''.join(open(LOG, encoding='utf-8').readlines()[-40:]))
    else:
        print(f'no log at {LOG} either -- did the run cell execute?')
else:
    print('outputs found in', OUT_ROOT)

In [ ]:
import json, glob, os, time
import pandas as pd

def iter_no(path):
    return int(path.rsplit('_', 1)[1].split('.')[0])

paths = sorted(glob.glob(os.path.join(OUT_ROOT, 'metrics_iter_*.json')), key=iter_no) if HAVE_OUTPUT else []
rows = []
for path in paths:
    m = json.load(open(path))
    rows.append({'iter': iter_no(path), 'accuracy': m.get('eval_accuracy'),
                 'f1_score': m.get('eval_f1_score'), 'loss': m.get('eval_loss')})
df = pd.DataFrame(rows)

if df.empty:
    print('no completed cycles -- check the run output above')
else:
    display(df)
    best = df.loc[df.f1_score.idxmax()]
    print(f"best: iter {int(best['iter'])}  f1={best['f1_score']:.4f}")
    stamps = [os.path.getmtime(p) for p in paths]
    if len(stamps) >= 2:
        print(f'{(stamps[-1]-stamps[0])/(len(stamps)-1)/60:.1f} min per cycle (measured)')

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(df['iter'], df['f1_score'], marker='o', label='macro F1')
    ax[0].plot(df['iter'], df['accuracy'], marker='s', label='accuracy')
    ax[0].axhline(0.10, ls='--', c='r', lw=1, label='collapse (F1=0.10)')
    ax[0].set_xlabel('AL iteration'); ax[0].set_ylabel('validation')
    ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(df['iter'], df['loss'], marker='o', c='purple')
    ax[1].axhline(1.3863, ls='--', c='r', lw=1, label='ln(4) = chance')
    ax[1].set_xlabel('AL iteration'); ax[1].set_ylabel('eval loss')
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.suptitle(f'{STRATEGY} -- {RUN_NAME}')
    plt.tight_layout(); plt.show()

In [ ]:
# Per-class behaviour on the last cycle + how balanced the acquired pool ended up
import glob, os
import pandas as pd

reports = sorted(glob.glob(os.path.join(OUT_ROOT, 'val_report_iter_*.csv')),
                 key=lambda p: int(p.rsplit('_', 1)[1].split('.')[0])) if HAVE_OUTPUT else []
if reports:
    print(os.path.basename(reports[-1]))
    display(pd.read_csv(reports[-1], index_col=0))
    print('HEALTHY  : all four classes have non-zero recall')
    print('COLLAPSED: accuracy 0.250 / macro-F1 0.100, one class holds every prediction')
else:
    print('no validation reports found')

In [ ]:
# Saved checkpoints and their test metrics
import json, os
import pandas as pd

if not HAVE_OUTPUT:
    raise SystemExit('no outputs -- see the diagnostic cell above')

best_meta = os.path.join(OUT_ROOT, 'best_model', 'best_iteration.json')
if os.path.exists(best_meta):
    print('best_model ->', json.load(open(best_meta)))

# 'best/' is only written when the best cycle was NOT the last one;
# otherwise 'final/' already describes the same weights.
for name in ('best', 'final'):
    root = os.path.join(OUT_ROOT, name)
    if not os.path.isdir(root):
        continue
    for split in sorted(os.listdir(root)):
        rep = os.path.join(root, split, 'total', 'classification_report.csv')
        if os.path.exists(rep):
            print(f'\n=== {name}/{split} ===')
            display(pd.read_csv(rep, index_col=0))

print('\nartefacts under', OUT_ROOT)
for entry in sorted(os.listdir(OUT_ROOT))[:40]:
    print('  ', entry)

## Notes

* **No resume.** A dead runtime restarts training from zero; completed cycles keep their
  metrics in Drive. Size runs to finish inside one session.
* **`best_model/` is ~1.1 GB, rewritten on every improvement** -- on a monotonic curve,
  every cycle. If Drive writes drag, point `output_folder` at `/content/` and copy out
  at the end.
* **`batch_size` is 32** to stay comparable with the bertin runs; 64 is faster on an
  A100 but changes the effective learning rate.
* **`warmup_steps` is 0**, matching bertin. If the first report shows eval loss pinned at
  1.386 and macro-F1 0.100, the run collapsed -- stop it and set `warmup_steps` to ~6% of
  the first cycle's steps.
* **Only `*_colab_fulldata.json` is row-for-row comparable to an HPC run**, because
  `BERTDataset` draws the subset itself. `prepare_colab_data.py` draws a different one.